In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import joblib
import ta  # Biblioteca de Análise Técnica
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [2]:
# Definimos Todas Variáveis
tickers = {
    'Bovespa': '^BVSP',        
    'Dolar': 'BRL=X',        
    'SP500': '^GSPC',        
    'Shanghai': '000001.SS', # Bolsa da China
    'Petroleo': 'BZ=F',      
    'Minerio': 'TIO=F',      
    'Ouro': 'GC=F',
    'Juros_BR': 'LFTS11.SA',
    'Inflacao_BR': 'IMAB11.SA'
}

In [3]:
print("📥 Baixando cotações históricas de 3 anos...")
df_precos = yf.download(list(tickers.values()), period='3y')['Close']
df_precos.rename(columns={v: k for k, v in tickers.items()}, inplace=True)
df_precos.ffill(inplace=True)
df_precos.dropna(inplace=True)

📥 Baixando cotações históricas de 3 anos...


[*********************100%***********************]  9 of 9 completed


In [4]:
print("🧬 Calculando Indicadores Técnicos e Retornos...")
df_features = pd.DataFrame(index=df_precos.index)

# A. Calculamos os retornos de todos os ativos
for col in df_precos.columns:
    df_features[col] = df_precos[col].pct_change()

# B. Injetamos o IFR / RSI (Momento)
df_features['RSI'] = ta.momentum.RSIIndicator(df_precos['Bovespa'], window=14).rsi()

# C. Injetamos a Tendência (Distância para a Média Móvel Simples de 15 dias)
sma_15 = ta.trend.SMAIndicator(df_precos['Bovespa'], window=15).sma_indicator()
df_features['Distancia_SMA15'] = (df_precos['Bovespa'] / sma_15) - 1

# D. Injetamos a Volatilidade (Largura das Bandas de Bollinger)
df_features['Bollinger_Width'] = ta.volatility.BollingerBands(df_precos['Bovespa'], window=20).bollinger_wband()

# Limpando os NaNs gerados pelos indicadores de média (primeiros 20 dias)
df_features.dropna(inplace=True)

time_steps = 7
colunas_features = df_features.columns

🧬 Calculando Indicadores Técnicos e Retornos...


In [5]:
def classificar_retorno_5_classes(retorno):
    if retorno <= -0.01:
        return 0  # Baixa Forte (<-1%)
    elif -0.01 < retorno <= -0.002:
        return 1  # Leve Baixa (-1% a -0.2%)
    elif -0.002 < retorno < 0.002:
        return 2  # Neutro (-0.2% a 0.2%)
    elif 0.002 <= retorno < 0.01:
        return 3  # Leve Alta (0.2% a 1%)
    else:
        return 4  # Alta Forte (>1%)

nomes_classes = {
    0: "📉 Baixa Forte (menor que -1.0%)",
    1: "↘️ Leve Baixa (-1.0% a -0.2%)",
    2: "➖ Neutro (-0.2% a +0.2%)",
    3: "↗️ Leve Alta (+0.2% a +1.0%)",
    4: "📈 Alta Forte (maior que +1.0%)"
}

In [6]:
# Criando o dataset com as janelas deslizantes (time_steps = 7)
def createDatasetClassificacao(dataset, time_steps):
    X, y = [], []
    idx_bovespa = list(dataset.columns).index('Bovespa')
    data_array = dataset.values
    
    for i in range(len(data_array) - time_steps):
        X.append(data_array[i:(i + time_steps)])
        retorno_futuro = data_array[i + time_steps, idx_bovespa]
        y.append(classificar_retorno_5_classes(retorno_futuro))
        
    return np.array(X), np.array(y)

X, y = createDatasetClassificacao(df_features, time_steps)

# One-Hot Encoding para o Alvo (Muda para num_classes=5)
y_encoded = to_categorical(y, num_classes=5)

# Divisão em Treino (80%) e Teste (20%)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y_encoded[:split], y_encoded[split:]

In [7]:
print("📏 Normalizando as Variáveis de Entrada...")
scaler_X = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_scaled = scaler_X.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

📏 Normalizando as Variáveis de Entrada...


In [8]:
print("🧠 Construindo a Rede Neural GRU...")
modelo_gru = Sequential([
    GRU(50, return_sequences=True, input_shape=(time_steps, len(colunas_features))),
    Dropout(0.2),
    GRU(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(5, activation='softmax') # 5 saídas com ativação Softmax para Probabilidades
])

modelo_gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

🧠 Construindo a Rede Neural GRU...


C:\Users\flavi\anaconda3\envs\SeriesTemporais\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
# Comando .fit() rodando por 60 épocas (assim como no seu modelo mais estável)
print("⏳ Treinando o modelo (Este processo estuda o histórico de 3 anos)...")
modelo_gru.fit(X_train_scaled, y_train, epochs=100, batch_size=32, validation_split=0.1, verbose=1)

⏳ Treinando o modelo (Este processo estuda o histórico de 3 anos)...
Epoch 1/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.2555 - loss: 1.5828 - val_accuracy: 0.2131 - val_loss: 1.5554
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2518 - loss: 1.5435 - val_accuracy: 0.3115 - val_loss: 1.5667
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2776 - loss: 1.5430 - val_accuracy: 0.3115 - val_loss: 1.5721
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2904 - loss: 1.5376 - val_accuracy: 0.3115 - val_loss: 1.5868
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2886 - loss: 1.5361 - val_accuracy: 0.3115 - val_loss: 1.5826
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2996 - loss: 1.5324 - val_accuracy: 0.3115 - val_loss: 1.5785
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2868 - loss: 1.5384 - val_accuracy: 0.3115 - val_loss: 1.5897
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s

In [10]:
if not os.path.exists('modelos'):
    os.makedirs('modelos')

# ATENÇÃO: Salvando com os nomes exatos que o seu Notebook Diário de 5 classes vai procurar!
modelo_gru.save_weights("modelos/modelo_gru_classificador.weights.h5", overwrite=True)
joblib.dump(scaler_X, 'modelos/scaler_X_classificador.pkl')

print("✅ Cérebro de 5 Classes treinado e salvo com sucesso na pasta 'modelos'!")

✅ Cérebro de 5 Classes treinado e salvo com sucesso na pasta 'modelos'!


In [11]:
loss, acuracia = modelo_gru.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n--- 📊 RESULTADOS DO TREINO (5 CLASSES COM TA) ---")
print(f"Acurácia Global do Modelo nos dados de Teste: {acuracia * 100:.2f}%")


--- 📊 RESULTADOS DO TREINO (5 CLASSES COM TA) ---
Acurácia Global do Modelo nos dados de Teste: 26.32%
